# Person 2 — Step 2: Download Thumbnails

Downloads high-res thumbnails from `thumbnail_url_high` with:
- Concurrent requests (fast)
- Skip-if-exists (safe to re-run)
- Progress bar
- A download manifest CSV for downstream notebooks

In [1]:
import sys
sys.path.append('../../')

import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

from shared.config import VIDEOS_CSV, CHANNELS_CSV, THUMBNAILS_DIR, PERSON2_DIR

In [2]:
videos   = pd.read_csv(VIDEOS_CSV)
channels = pd.read_csv(CHANNELS_CSV)[['channel_id', 'channel_title', 'group_label']]
df = videos.merge(channels, on='channel_id')

# Keep all videos that have at least one thumbnail URL
df = df[df['thumbnail_url_high'].notna() | df['thumbnail_url_default'].notna()].copy()
print(f"{len(df)} videos with thumbnail URLs")

9569 videos with thumbnail URLs


In [5]:
def fallback_urls(video_id: str) -> list[str]:
    """YouTube thumbnail resolutions in decreasing quality order."""
    base = f"https://i.ytimg.com/vi/{video_id}"
    return [
        f"{base}/maxresdefault.jpg",
        f"{base}/hqdefault.jpg",
        f"{base}/mqdefault.jpg",
        f"{base}/sddefault.jpg",
        f"{base}/default.jpg",
    ]

def download_one(row):
    video_id = row['video_id']
    dest     = THUMBNAILS_DIR / f"{video_id}.jpg"

    if dest.exists():
        return video_id, 'skipped'

    # Build URL list: stored high → stored default → all standard resolutions
    stored = [row.get('thumbnail_url_high'), row.get('thumbnail_url_default')]
    urls_to_try = [u for u in stored if pd.notna(u)] + fallback_urls(video_id)

    # Deduplicate while preserving order
    seen, unique_urls = set(), []
    for u in urls_to_try:
        if u not in seen:
            seen.add(u)
            unique_urls.append(u)

    for url in unique_urls:
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            dest.write_bytes(r.content)
            label = url.split('/')[-1]
            suffix = '' if url == row.get('thumbnail_url_high') else f' (fallback: {label})'
            return video_id, f'ok{suffix}'
        except requests.HTTPError as e:
            if e.response.status_code == 404:
                continue
            return video_id, f'error: {e}'
        except Exception as e:
            return video_id, f'error: {e}'

    return video_id, 'error: no thumbnail available at any resolution'

rows = df.to_dict('records')
results = {}

with ThreadPoolExecutor(max_workers=20) as pool:
    futures = {pool.submit(download_one, row): row['video_id'] for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Downloading thumbnails'):
        vid, status = future.result()
        results[vid] = status

status_series = pd.Series(results).value_counts()
print(status_series)

skipped                                            9568
error: no thumbnail available at any resolution       1
Name: count, dtype: int64


In [4]:
# Save manifest so downstream notebooks know which files exist
manifest = (
    df[['video_id', 'channel_title', 'group_label', 'view_count', 'like_count', 'comment_count', 'duration_seconds', 'published_at']]
    .assign(
        download_status=lambda d: d['video_id'].map(results),
        thumb_path=lambda d: d['video_id'].apply(lambda v: str(THUMBNAILS_DIR / f"{v}.jpg"))
    )
)
manifest_path = PERSON2_DIR / 'outputs' / 'thumbnail_manifest.csv'
manifest.to_csv(manifest_path, index=False)
print(f"Manifest saved → {manifest_path}")
manifest[manifest['download_status'] == 'ok'].shape

Manifest saved → /Users/alexlag/Documents/MA2/CompSocialMedia/YouTubeCompare/analysis/person2_titles_thumbnails/../../analysis/person2_titles_thumbnails/outputs/thumbnail_manifest.csv


(0, 10)